In [ ]:
# Databricks notebook source
# tutorial_name: 04-Day8-Medallion-Silver-Task
# notebookName: 04-Day8-Medallion-Silver-Task
# COMMAND ----------

# Databricks notebook source
# COMMAND ----------

# DBTITLE 1,Paths (same as Days 5–8)
notepoint_rel = "hands-on/day-08/notebooks/04-Day8-Medallion-Silver-Task.ipynb"
BASE_PATH = "abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/"
STUDENT_ID = "u25"  # u01–u16
DAY5 = BASE_PATH + "/day5"
P_BASIC = DAY5 + "/delta/flight_summary_basic"
SOURCE_CSV = BASE_PATH + "/2010-summary.csv"
SOURCE_JSON = BASE_PATH + "/json/2015-summary.json"
DAY8_ROOT = f"{BASE_PATH}day08-{STUDENT_ID}"
MED_ROOT = f"{DAY8_ROOT}/medallion"
BRONZE_PATH = f"{MED_ROOT}/bronze_flight_routes"
SILVER_PATH = f"{MED_ROOT}/silver_flight_routes"
GOLD_PATH = f"{MED_ROOT}/gold_by_destination"
METRICS_PATH = f"{DAY8_ROOT}/workflow_metrics/run_metrics"
# COMMAND ----------
print("DAY8_ROOT:", DAY8_ROOT)
print("BRONZE_PATH:", BRONZE_PATH)
print("SILVER_PATH:", SILVER_PATH)
print("GOLD_PATH:", GOLD_PATH)
try:
    spark.read.format("delta").load(P_BASIC).limit(1).collect()
    print("OK: P_BASIC")
except Exception as e:
    print("P_BASIC:", e)
try:
    spark.read.format("csv").option("header", True).load(SOURCE_CSV).limit(1).collect()
    print("OK: SOURCE_CSV")
except Exception as e:
    print("SOURCE_CSV:", e)
try:
    spark.read.format("json").load(SOURCE_JSON).limit(1).collect()
    print("OK: SOURCE_JSON (optional)")
except Exception as e:
    print("(optional) SOURCE_JSON:", type(e).__name__)
# COMMAND ----------


## Silver task (item 22)

**Purpose:** Read **Bronze** Delta, drop rows with null **destination** or **origin** country, add **`silver_built_ts`**, write **Silver** Delta.

**Requires:** Bronze table already exists at **`BRONZE_PATH`** (run **03** or job task `bronze` first).

**Output:** Overwrites **`SILVER_PATH`**.


In [ ]:
from pyspark.sql import functions as F

df = spark.read.format("delta").load(BRONZE_PATH)
out = df.filter(
    F.col("DEST_COUNTRY_NAME").isNotNull() & F.col("ORIGIN_COUNTRY_NAME").isNotNull()
)
out = out.withColumn("silver_built_ts", F.current_timestamp())
out.write.format("delta").mode("overwrite").save(SILVER_PATH)
print("Silver rows:", out.count(), "->", SILVER_PATH)
spark.read.format("delta").load(SILVER_PATH).limit(5).show(truncate=False)
# COMMAND ----------


## Null key counts on Bronze (why Silver filters)

In [ ]:
try:
    b = spark.read.format("delta").load(BRONZE_PATH)
    b.select(
        F.sum(F.when(F.col("DEST_COUNTRY_NAME").isNull(), 1).otherwise(0)).alias("null_dest"),
        F.sum(F.when(F.col("ORIGIN_COUNTRY_NAME").isNull(), 1).otherwise(0)).alias("null_origin"),
        F.count(F.lit(1)).alias("rows"),
    ).show()
except Exception as e:
    print("Run bronze first:", e)
# COMMAND ----------
